# Snake RL — Training Run Analysis

Analyzes training runs logged to MLflow and documents reconstructed notes from runs where data was not captured.

**Runs summary:**
| Run | Grid (playable) | food | collision | ent_coef | toward | Outcome |
|-----|----------------|------|-----------|----------|--------|--------|
| 1 | 24×24 (total) | 64 | -8 | 0.05 | 0.10 | Shaping local optimum — wandered, no food |
| 2 | 14×14 (16×16 total, border bug) | 64 | -8 | 0.05 | 0.01 | Same local optimum, entropy collapsed |
| 3 | 16×16 (18×18 total) | 16 | -2 | 0.10 | 0.01 | Entropy stuck at max — policy stayed random |
| 4 | 16×16 | 16 | -2 | 0.02 | 0.0 | Entropy in target zone (~-0.6), reward flat 2–5 — PPO structural failure; switching to DQN |

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..') / 'src'))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import Image

sns.set_theme(style='darkgrid', palette='tab10')
plt.rcParams['figure.dpi'] = 120

try:
    import mlflow
    from mlflow.tracking import MlflowClient
    DB_PATH = (Path('..') / 'mlruns' / 'mlflow.db').resolve()
    TRACKING_URI = f'sqlite:///{DB_PATH}'
    mlflow.set_tracking_uri(TRACKING_URI)
    client = MlflowClient(TRACKING_URI)
    exp = client.get_experiment_by_name('snake-rl')
    if exp is None:
        raise ValueError("Experiment 'snake-rl' not found in MLflow DB")
    print(f'Tracking URI: {TRACKING_URI}')
    print(f'Experiment ID: {exp.experiment_id}')
except Exception as e:
    print(f'MLflow unavailable: {e}')
    print('Static PNGs will be displayed instead of live charts.')
    client = None
    exp = None

## Reconstructed notes: Runs 1 & 2

MLflow data was not captured for Runs 1 and 2. The following is reconstructed from observations during training.

### Run 1 — 24×24 grid, toward=0.10 (pre-2026-03-22)

**Config:** `grid=24×24 (total)`, `food=64`, `collision=-8`, `ent_coef=0.05`, `toward=0.10`, `away=-0.05`, `max_steps=500`, `lr=0.0003`

| Checkpoint | ep_rew_mean | ep_len_mean | explained_variance | entropy_loss | Notes |
|-----------|------------|------------|-------------------|-------------|-------|
| ~500k | ~20 | ~200 | — | — | Healthy early progress |
| ~925k | ~30 | ~314 | 0.32 | — | Plateau confirmed; value_loss=19.8, approx_kl=0.023 |
| ~1.2–1.3M | ~30.7 | ~314 | — | — | Best checkpoint |
| ~2.1M | 11–17 | — | 0.12–0.33 | -0.97 | Regression; entropy collapsed; policy deterministic |

**Diagnosis:** `toward=0.10` over 500 steps ≈ 30–50 reward/episode from shaping alone, comparable to food=64 reward. Agent optimized shaping instead of eating food. Live preview showed yellow borders (timeout), not red (collision) — agent learned to survive by wandering. Entropy collapsed to -0.97 by 2.1M steps.

**Key insight:** When shaping reward is large enough to substitute for food reward, the agent never needs to eat.

---

### Run 2 — 16×16 grid (14×14 playable due to border bug), toward=0.01 (2026-03-22)

**Config:** `grid=16×16 total (14×14 playable)`, `food=64`, `collision=-8`, `ent_coef=0.05`, `toward=0.01`, `away=-0.02`, `max_steps=300`, `lr=0.0001`

| Checkpoint | ep_rew_mean | ep_len_mean | explained_variance | entropy_loss | Notes |
|-----------|------------|------------|-------------------|-------------|-------|
| 3.1M (stopped) | 10.4 | 138 | 0.517 | -0.873 | Plateau; entropy declining; zero food being eaten |

**Diagnosis:** With `food=64` and `ep_rew_mean=10.4`, all reward is shaping (138 steps × 0.01 ≈ 1 reward from toward signal — the rest is structure noise). The agent ate essentially zero food across 3.1M steps. Entropy was collapsing toward determinism. Live preview: single-cell snake (no growth) never near food. `value_loss=41.6` — high variance in value function due to food=64 scale. Also affected by border bug: config said 16×16 but playable space was only 14×14.

**Key insight:** `collision=-8` was expensive enough that the agent learned risk-aversion before discovering food. `food=64` caused high value-function variance.

## Load MLflow runs

In [ ]:
def get_metric_df(run_id: str, metric: str) -> pd.DataFrame:
    """Return a DataFrame with columns [step, value] for a given metric."""
    if client is None:
        return pd.DataFrame(columns=['step', metric])
    history = client.get_metric_history(run_id, metric)
    if not history:
        return pd.DataFrame(columns=['step', metric])
    return pd.DataFrame({'step': [h.step for h in history], metric: [h.value for h in history]})


if client is not None:
    def get_run_summary(run) -> dict:
        import datetime
        p = run.data.params
        m = run.data.metrics
        return {
            'run_id': run.info.run_id[:8],
            'start': datetime.datetime.fromtimestamp(run.info.start_time / 1000).strftime('%Y-%m-%d %H:%M'),
            'status': run.info.status,
            'food': float(p.get('reward.food', 'nan')),
            'collision': float(p.get('reward.collision', 'nan')),
            'toward': float(p.get('reward.toward', 'nan')),
            'ent_coef': float(p.get('ppo.ent_coef', 'nan')),
            'lr': float(p.get('ppo.learning_rate', 'nan')),
            'grid': f"{p.get('env.grid_w')}×{p.get('env.grid_h')}",
            'max_steps': int(p.get('env.max_steps', 0)),
            'final_ep_rew': m.get('ep_rew_mean', float('nan')),
            'final_ep_len': m.get('ep_len_mean', float('nan')),
            'final_entropy': m.get('entropy', float('nan')),
            'final_value_loss': m.get('value_loss', float('nan')),
        }
    all_runs = client.search_runs(exp.experiment_id, order_by=['start_time ASC'])
    summaries = [get_run_summary(r) for r in all_runs]
    summary_df = pd.DataFrame(summaries)
    print(f'{len(all_runs)} run(s) found in MLflow')
    display(summary_df)
else:
    print('MLflow data not available — see saved PNGs below.')
    all_runs = []

## Run 3 — Full metric curves

In [ ]:
# Run 3 metric plots — saved from original analysis (MLflow data no longer available)
Image('run3_metrics.png')

## Run 3 — Diagnosis

In [ ]:
# Run 3 diagnosis — values computed from original MLflow data (now deleted), preserved here for reference.
max_entropy_3actions = np.log(3)  # ≈ 1.099 nats

print('=== Run 3 Diagnosis ===')
print()
print('Entropy (|entropy_loss|):')
print(f'  Max possible (ln 3):  {max_entropy_3actions:.4f} nats')
print(f'  Run 3 start (~16k):   1.0979')
print(f'  Run 3 end (5M):       1.0109')
print(f'  → Policy was 91% of max entropy on average = essentially RANDOM throughout')
print()
print('Episode reward:')
print('  food reward = 16.0 per food eaten')
print('  Final ep_rew_mean: 2.915')
print('  Max ep_rew_mean seen: 7.216 @ step 901,120')
print('  Estimated foods/episode: ~0.15 (after subtracting shaping estimate of 0.49)')
print()
print('Root cause: ent_coef=0.10 too high.')
print('The entropy bonus dominated the policy gradient.')
print('The policy could not commit to any learned behavior.')

## Cross-run comparison (reconstructed + MLflow)

In [ ]:
colors = sns.color_palette('tab10', 4)

reconstructed = pd.DataFrame([
    {
        'run': 'Run 1\n(reconstructed)',
        'food': 64.0, 'collision': -8.0, 'toward': 0.10, 'ent_coef': 0.05,
        'grid': '22×22\n(24 total)', 'max_steps': 500, 'lr': 0.0003,
        'peak_ep_rew': 30.7, 'final_ep_rew': 14.0,
        'final_ep_len': 314, 'final_entropy': -0.97,
        'stopped_at': 2_100_000,
        'outcome': 'Shaping local optimum\n(toward≈food reward)'
    },
    {
        'run': 'Run 2\n(reconstructed)',
        'food': 64.0, 'collision': -8.0, 'toward': 0.01, 'ent_coef': 0.05,
        'grid': '14×14\n(16 total, border bug)', 'max_steps': 300, 'lr': 0.0001,
        'peak_ep_rew': 10.4, 'final_ep_rew': 10.4,
        'final_ep_len': 138, 'final_entropy': -0.87,
        'stopped_at': 3_100_000,
        'outcome': 'Risk-aversion local optimum\n(collision too costly)'
    },
    {
        'run': 'Run 3\n(MLflow)',
        'food': 16.0, 'collision': -2.0, 'toward': 0.01, 'ent_coef': 0.10,
        'grid': '16×16\n(18 total)', 'max_steps': 300, 'lr': 0.0001,
        'peak_ep_rew': 4.4, 'final_ep_rew': 2.9,
        'final_ep_len': 98.5, 'final_entropy': -1.01,
        'stopped_at': 5_000_000,
        'outcome': 'ent_coef too high\n(policy stayed random)'
    },
    {
        'run': 'Run 4\n(reconstructed)',
        'food': 16.0, 'collision': -2.0, 'toward': 0.0, 'ent_coef': 0.02,
        'grid': '16×16', 'max_steps': 300, 'lr': 0.0001,
        'peak_ep_rew': 5.0, 'final_ep_rew': 3.5,
        'final_ep_len': 150, 'final_entropy': -0.60,
        'stopped_at': 5_000_000,
        'outcome': 'PPO structural failure\n(sparse reward, on-policy)'
    },
])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Cross-Run Comparison — All 4 PPO Runs (Runs 1, 2, 4 reconstructed from notes)', fontsize=13)

runs = reconstructed['run'].tolist()

axes[0].bar(runs, reconstructed['final_ep_rew'], color=colors)
axes[0].set_title('Final ep_rew_mean')
axes[0].set_ylabel('Mean Episode Reward')
axes[0].axhline(0, color='black', linewidth=0.5)
for i, v in enumerate(reconstructed['final_ep_rew']):
    axes[0].text(i, v + 0.3, f'{v:.1f}', ha='center', fontsize=9)

axes[1].bar(runs, reconstructed['final_entropy'].abs(), color=colors)
axes[1].axhline(np.log(3), color='red', linestyle='--', alpha=0.7, label=f'max entropy = ln(3)≈1.10')
axes[1].axhspan(0.5, 0.7, alpha=0.15, color='green', label='target zone [0.5, 0.7]')
axes[1].set_title('Final |entropy| (higher = more random)')
axes[1].set_ylabel('|entropy_loss|')
axes[1].legend(fontsize=8)
for i, v in enumerate(reconstructed['final_entropy'].abs()):
    axes[1].text(i, v + 0.01, f'{v:.2f}', ha='center', fontsize=9)

axes[2].bar(runs, reconstructed['ent_coef'], color=colors)
axes[2].set_title('ent_coef per run')
axes[2].set_ylabel('ent_coef')
for i, v in enumerate(reconstructed['ent_coef']):
    axes[2].text(i, v + 0.001, f'{v}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('cross_run_comparison.png', bbox_inches='tight')
plt.show()

print('\nRun outcomes:')
for _, row in reconstructed.iterrows():
    print(f'  {row["run"].replace(chr(10), " ")}: {row["outcome"].replace(chr(10), " ")}')

## Entropy Goldilocks analysis

All four runs failed — three for opposite entropy-related reasons, and one confirming the entropy hypothesis was a red herring:

| Run | ent_coef | Final entropy | Failure mode |
|-----|----------|---------------|--------------|
| 1 | 0.05 | -0.97 (low) | Collapsed — deterministic "avoid walls" policy |
| 2 | 0.05 | -0.87 (low) | Collapsing — same deterministic local optimum forming |
| 3 | 0.10 | -1.01 (≈max) | Stuck at max — policy gradient overwhelmed by entropy bonus |
| 4 | 0.02 | -0.60 (target zone) | **Correct entropy, still failed** — PPO structural issue |

**Run 4 was the decisive experiment.** Entropy landed squarely in the target zone and reward was still flat at 2–5. This rules out entropy tuning as root cause.

**True root cause:** PPO is on-policy. With `max_steps=300` and food sparsely placed on a 16×16 grid, a random agent eats food roughly once per ~50 steps on average. Each PPO rollout (2048 steps × 4 envs = 8192 total) discards those experiences after one gradient update. The Q-value signal for "being near food is good" never accumulates.

**Fix:** Switch to DQN. A replay buffer of 100k transitions lets rare food-eating events be replayed many times. Double DQN reduces overestimation of the "safe wandering" state value.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

ent_coefs = [0.05, 0.05, 0.10, 0.02]
entropies = [0.97, 0.87, 1.01, 0.60]  # absolute values
labels_pts = ['Run 1\n(2.1M steps)', 'Run 2\n(3.1M steps)', 'Run 3\n(5M steps)', 'Run 4\n(5M steps)']

ax.scatter(ent_coefs, entropies, s=150, zorder=5, color=colors)
for i, (x, y, lbl) in enumerate(zip(ent_coefs, entropies, labels_pts)):
    ax.annotate(lbl, (x, y), textcoords='offset points', xytext=(10, 5), fontsize=9)

# Target zone
ax.axhspan(0.5, 0.7, alpha=0.15, color='green', label='Target entropy zone [0.5, 0.7]')
ax.axhline(np.log(3), color='red', linestyle='--', alpha=0.5, label='Max entropy = ln(3)')
ax.axhline(0, color='black', linestyle='--', alpha=0.3, label='Fully deterministic')

ax.set_xlabel('ent_coef', fontsize=11)
ax.set_ylabel('Final |entropy| (nats)', fontsize=11)
ax.set_title('ent_coef vs Final Entropy — Run 4 landed in target zone but still failed', fontsize=12)
ax.legend(fontsize=9)
ax.set_xlim(-0.01, 0.13)
ax.set_ylim(-0.05, 1.25)

plt.tight_layout()
plt.savefig('entropy_goldilocks.png', bbox_inches='tight')
plt.show()

## Run 4 — Reconstructed notes (no MLflow data)

In [ ]:
print("""
Config (from default.yaml at time of run):
  grid:         16×16 (18×18 total with 1-cell border)
  food:         16.0
  collision:    -2.0
  toward:       0.0   (shaping removed)
  away:         0.0   (shaping removed)
  ent_coef:     0.02  (target: entropy in [-0.7, -0.5])
  lr:           0.0001
  n_envs:       4
  total_steps:  5,000,000

Observed metrics (from Streamlit live dashboard):
  ep_rew_mean:  ~2–5 (flat throughout)
  ep_len_mean:  ~100–200
  entropy:      ~-0.5 to -0.7  ← IN TARGET ZONE
  value_loss:   ~0–4 (noisy, no trend)
  Status:       FINISHED (5M steps)

Diagnosis:
  Entropy was correctly tuned — ent_coef=0.02 kept policy in the productive
  exploration range. Despite this, ep_rew_mean never trended upward.
  The snake survived but did not eat.

  This confirms: entropy tuning was NOT the root cause of prior failures.
  PPO is structurally ill-suited to this problem. As an on-policy algorithm,
  it discards experience after each rollout (~8192 steps). With food sparsely
  placed on a 16×16 grid, only a handful of food-eating transitions appear in
  each rollout — too few for the policy gradient to credit-assign reliably.

  Decision: Switch to DQN (off-policy, replay buffer, Double DQN by default).
""")

## Decision — Switch to DQN

**Algorithm:** Double DQN (default in SB3's `DQN`) with replay buffer.

**Why DQN wins here:**

| Property | PPO (runs 1–4) | DQN (run 5+) |
|---|---|---|
| Data efficiency | Discards rollout after 1 update | Replays each transition many times |
| Sparse reward handling | Weak gradient from rare food events | PER-style or large buffer amplifies rare events |
| Credit assignment | GAE over rollout window | TD bootstrapping propagates Q-values backward |
| Exploration | `ent_coef` balancing act | Epsilon-greedy: explicit, decoupled from learning |
| Overestimation | N/A | Double DQN corrects target Q overestimation |

**Key DQN config (run 5):**
```
buffer_size:              100,000
learning_starts:          10,000
exploration_fraction:     0.3   (epsilon 1.0 → 0.05 over 1.5M steps)
exploration_final_eps:    0.05
train_freq:               4
target_update_interval:   1,000
```

**What success looks like:** `ep_rew_mean` trending upward past 16 (≥1 food/episode) within ~1M steps.

## Helper: load any future DQN run

Re-run this cell after starting a DQN run to plot its metric curves.

In [ ]:
def plot_dqn_run_curves(run_id_prefix: str, label: str = None):
    """Plot metric curves for a DQN run by its MLflow run ID prefix."""
    if client is None:
        print('MLflow unavailable — cannot load run curves.')
        return
    all_r = client.search_runs(exp.experiment_id)
    matches = [r for r in all_r if r.info.run_id.startswith(run_id_prefix)]
    if not matches:
        print(f'No run found with prefix {run_id_prefix!r}')
        return
    run = matches[0]
    label = label or run.info.run_id[:8]

    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    fig.suptitle(f'DQN Run: {label}', fontsize=12)

    metrics = [
        ('ep_rew_mean',       'Mean Episode Reward'),
        ('ep_len_mean',       'Mean Episode Length'),
        ('loss',              'TD Loss'),
        ('exploration_rate',  'Epsilon (exploration rate)'),
    ]

    for ax, (metric, ylabel) in zip(axes, metrics):
        df = get_metric_df(run.info.run_id, metric)
        if df.empty:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(ylabel)
            continue
        ax.plot(df['step'], df[metric], alpha=0.4, linewidth=0.8)
        if len(df) > 20:
            smoothed = df[metric].rolling(window=max(1, len(df)//20), center=True).mean()
            ax.plot(df['step'], smoothed, linewidth=2)
        ax.set_xlabel('Steps')
        ax.set_ylabel(ylabel)
        ax.set_title(ylabel)
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
        if metric == 'ep_rew_mean':
            food = float(run.data.params.get('reward.food', 16))
            ax.axhline(food, color='green', linestyle='--', alpha=0.5, label=f'1 food ({food})')
            ax.legend(fontsize=8)

    plt.tight_layout()
    plt.show()


# Example (fill in run ID prefix after starting training):
# plot_dqn_run_curves('abc12345', label='Run 5 — DQN baseline')